In [8]:
import pandas as pd
data=pd.read_csv('JAIS_2025_article_data.csv')
data.head()


,Title,Year,Issue,ArticleURL,Abstract,Author_Name,Author_University
0,Legal Compliance and the Open Texture of Law,2025,Issue 1,https://aisel.aisnet.org/jais/vol26/iss1/10,No Abstract,Arto Lanamäki,University of Oulu
1,Legal Compliance and the Open Texture of Law,2025,Issue 1,https://aisel.aisnet.org/jais/vol26/iss1/10,No Abstract,Mika Viljanen,University of Turku
2,Legal Compliance and the Open Texture of Law,2025,Issue 1,https://aisel.aisnet.org/jais/vol26/iss1/10,No Abstract,Karin Väyrynen,University of Oulu
3,Legal Compliance and the Open Texture of Law,2025,Issue 1,https://aisel.aisnet.org/jais/vol26/iss1/10,No Abstract,Lyria Bennett Moses,UNSW Sydney
4,Ecological Validity in NeuroIS Research: Theor...,2025,Issue 1,https://aisel.aisnet.org/jais/vol26/iss1/9,No Abstract,Ali Balapour,Northern Kentucky University


In [9]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 161 entries, 0 to 160
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Title              161 non-null    object
 1   Year               161 non-null    int64 
 2   Issue              161 non-null    object
 3   ArticleURL         161 non-null    object
 4   Abstract           161 non-null    object
 5   Author_Name        161 non-null    object
 6   Author_University  161 non-null    object
dtypes: int64(1), object(6)
memory usage: 8.9+ KB


In [10]:
data.isnull().sum()

Title                0
Year                 0
Issue                0
ArticleURL           0
Abstract             0
Author_Name          0
Author_University    0
dtype: int64

In [12]:
data.duplicated().sum()

0

In [18]:
data = data.where(pd.notna(data), None)


In [17]:
data.iloc[[872]]


,Title,Year,Issue,ArticleURL,Abstract,Author_Name,Author_University
872,(Re)considering the Concept of Literature Revi...,2020,Issue 5,https://aisel.aisnet.org/jais/vol21/iss5/10,No Abstract,Guy Pare,HEC Montr√©al


In [5]:
print(data.info())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 161 entries, 0 to 160
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Title              161 non-null    object
 1   Year               161 non-null    int64 
 2   Issue              161 non-null    object
 3   ArticleURL         161 non-null    object
 4   Abstract           161 non-null    object
 5   Author_Name        161 non-null    object
 6   Author_University  161 non-null    object
dtypes: int64(1), object(6)
memory usage: 8.9+ KB
None


In [13]:
print(data.isnull().sum())
print(data.describe())
data=data.fillna('Null')

Title                0
Year                 0
Issue                0
ArticleURL           0
Abstract             0
Author_Name          0
Author_University    0
dtype: int64
         Year
count   161.0
mean   2025.0
std       0.0
min    2025.0
25%    2025.0
50%    2025.0
75%    2025.0
max    2025.0


In [14]:
data.duplicated().sum()


0

In [34]:
duplicates = data[data.duplicated(subset=['Author_Name','ArticleURL'], keep='first')]
print(duplicates)

Empty DataFrame
Columns: [Title, Year, Issue, ArticleURL, Abstract, Author_Name, Author_University]
Index: []


In [15]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 161 entries, 0 to 160
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Title              161 non-null    object
 1   Year               161 non-null    int64 
 2   Issue              161 non-null    object
 3   ArticleURL         161 non-null    object
 4   Abstract           161 non-null    object
 5   Author_Name        161 non-null    object
 6   Author_University  161 non-null    object
dtypes: int64(1), object(6)
memory usage: 8.9+ KB


In [ ]:
import openai
import pandas as pd
import json
import csv
openai_api_key = os.getenv("OPENAI_API_KEY")
# print(openai_api_key)

def extract_author_address(author_address):
    # prompt=f"""
    # Extract the following details from this web journal articles url
    # URL	Author_Name	Standardized_Name	Standardized_University	Author_University	Author_Department	Author_State	Author_Country	Author_Pincode	title
    # # If any detail is missing, return NULL for that field and make sure maintain the same name for field values to search by that. 
    # # ex: in one record US and other record USA and other record contains United States of america display common name for these 3 record values. Same for university, department, state names also if Wichita State University in one record and WSU in other record display common name to use in search operation in my website. 
    # # If it is common name I can search easily.
    
    # # URL: {address}
    
    # # Return the data in JSON format.
    # """

   prompt = f"""
        Extract, standardize, and enrich the following author and university information. 
        Your task is to ensure consistent, deduplicated, and accurate naming across all data.

        IMPORTANT REQUIREMENTS:
        1. **Author Names**
        - Convert all names to English alphabet (transliterate non-English characters).
        - Apply title case (First Last).
        - Standardize across variations:
            - "P Prakash" → "Ponduri Prakash"
            - "Dr. John Smith" → "John Smith"
            - "Smith, John" → "John Smith"
            - "J. Smith" and "John Smith" → "John Smith"
        - Ensure one unique standardized name appears everywhere for the same person.

        2. **University Names**
        - Standardize and unify variations to the official English name.
            - Example: "WSU", "Wichita Statte University", "Wichita State University USA" 
            → "Wichita State University"
        - Translate foreign-language university names to their official English equivalent.
        - Always return the same name for the same university across records.

        3. **Location Data (State & Country)**
        - Standardize country names to official English (e.g., US/USA → "United States").
        - Standardize state/province names (e.g., CA → "California").
        - If missing, determine the correct **state and country** for the given university 
            using external knowledge (Google, world university data).
        - Ensure consistency: the same university must always map to the same state and country.

       
        CRITICAL:
        - Return only a single JSON object (not an array).
        - Use English-only text in final output.
        - Make sure author, university, state, and country are 100% consistent across all rows.

        Text to process: {author_address}

        Output format:
        {{
            "Author": "Ponduri Prakash",
            "Standardized_Author": "Ponduri Prakash", 
            "University": "Wichita State University",
            "Department": "Computer Science",
            "State": "Kansas",
            "Country": "United States",
            "Pincode": "67260"
        }}"""
   response=openai.chat.completions.create(
        model='gpt-3.5-turbo',
        messages=[
            {"role":"system","content":"You are an expert in structured data extraction."},
            {"role":"user","content":prompt}
        ],
        temperature=0
    )
   try:
        extracted_data=response.choices[0].message.content.strip().strip('```json').strip('```')
        structured_data=json.loads(extracted_data)
        author_data={
        'Standardized_Author': structured_data.get('Standardized_Author', None),
        'University': structured_data.get('University', None),
        # 'Department': structured_data.get('Department', None),
        'State': structured_data.get('State', None),
        'Country': structured_data.get('Country', None),
        # 'Pincode': structured_data.get('Pincode', None) # assuming pincode is always present in the structured data. If not, return None.
        }
        return author_data 
   except Exception as e:
        print(e)
   return None
   


with open('extract_2025.csv', mode='a', newline='', encoding='utf-8') as file:
    writer = csv.writer(file)
    writer.writerow(['URL','Journal_Title','Article_Title','Abstract','Author_name','Standardized_Author','Author_University','Author_State','Author_Country'])
for index,row in data.iterrows():
    address=row['Author_Name']+row['Author_University']
    result = extract_author_address(address)
    if not result:
        continue
    with open('extract_2025.csv', mode='a', newline='',encoding='utf-8') as file:
        writer=csv.writer(file)
        writer.writerow([row['ArticleURL'],'Journal of the Association for Information Systems',row['Title'],row['Abstract'],row['Author_Name'],result['Standardized_Author'],result['University'],result['State'],result['Country']])

In [2]:
jais_data=pd.read_csv('extract.csv')
jais_data.head()

,URL,Journal_Title,Article_Title,Abstract,Author_name,Standardized_Author,Author_University,Author_State,Author_Country,Unnamed: 9,...,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24
0,https://aisel.aisnet.org/jais/vol25/iss1/15,Journal of the Association for Information Sys...,A Knowledge Management Perspective of Generati...,No Abstract,Maryam Alavi,Maryam Alavi,Georgia Institute of Technology,Georgia,United States,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,https://aisel.aisnet.org/jais/vol25/iss1/15,Journal of the Association for Information Sys...,A Knowledge Management Perspective of Generati...,No Abstract,Dorothy E. Leidner,Dorothy E. Leidner,University of Virginia,Virginia,United States,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,https://aisel.aisnet.org/jais/vol25/iss1/15,Journal of the Association for Information Sys...,A Knowledge Management Perspective of Generati...,No Abstract,Reza Mousavi,Reza Mousavi,University of Virginia,Virginia,United States,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,https://aisel.aisnet.org/jais/vol25/iss1/14,Journal of the Association for Information Sys...,The Societal Impacts of Generative Artificial ...,No Abstract,Rajiv Sabherwal,Rajiv Sabherwal,University of Arkansas,Arkansas,United States,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,https://aisel.aisnet.org/jais/vol25/iss1/14,Journal of the Association for Information Sys...,The Societal Impacts of Generative Artificial ...,No Abstract,Varun Grover,Varun Grover,University of Arkansas,Arkansas,United States,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
jais_data = jais_data.where(pd.notna(jais_data), None)
jais_data=jais_data.fillna('Null')

In [4]:
jais_data.duplicated().sum()

0

In [14]:
import pandas as pd
import mysql.connector
import json

# Connect to MySQL
#Used the DSS_new for date matching
db = mysql.connector.connect(
        host=  "127.0.0.1",
        user= "root",
        password="Nanna&143",
        database="journals",
        auth_plugin='mysql_native_password')

cursor = db.cursor()
# Insert data into MySQL
for index, row in jais_data.iterrows():
    articleUrl=row["URL"] if row['URL'] else None
    journal_title=row["Journal_Title"] if row['Journal_Title'] else None
    article_title=row["Article_Title"] if row['Article_Title'] else None
    # abstract=row["Abstract"] if row['Abstract'] else None
    # Year=row["Year"] if row['Year'] else None
    Author_University=row["Author_University"] if row['Author_University'] else None
    Author_Name=row["Author_name"] if row['Author_name'] else None
    standardized_author=row["Standardized_Author"] if row['Standardized_Author'] else None
    # author_email=row["Author_email"] if row['Author_email'] else None
    # author_address=row["Author_Address"] if row['Author_Address'] else None
    author_university=row["Author_University"] if row['Author_University'] else None
    # author_department=row["Author_department"] if row['Author_department'] else None
    author_state=row["Author_State"] if row['Author_State'] else None  #   
    author_country=row["Author_Country"] if row['Author_Country'] else None
        
    try:
        sql = """
        INSERT INTO JAIS_DATA(URL,Journal_Title,Article_Title,Author_Name,Standardized_Author,Author_University,Author_State,Author_Country) values(%s, %s, %s, %s, %s, %s, %s, %s)
        """
        values = (
            articleUrl,
            journal_title,
            article_title,
            Author_Name,
            standardized_author,
            author_university,
            author_state,
            author_country

        )
        
        cursor.execute(sql, values)
    except Exception as e:
        print(f"Error inserting data: {e}{row["URL"]}")
        break
# Commit & close connection
db.commit()
cursor.close()
db.close()

print("Data inserted successfully!")

Data inserted successfully!


In [9]:
#For the decision support systems titles
import pandas as pd
import mysql.connector
import json

#connect to mysql
db=mysql.connector.connect(
    host="127.0.0.1",
    user='root',
    password='Nanna&143',
    database='journals',
    auth_plugin='mysql_native_password'
)
cursor=db.cursor()
for index, row in df.iterrows():
    title = row["Title"] if row["Title"] else None
    url = row["URL"] if row["URL"] else None
    vol_year=row["Year"] if row["Year"] else None

    try:
        sql = """
        INSERT INTO JSIS_Articles
        (Article_Title, URL, Month_Year)
        VALUES (%s, %s, %s)
        """
        values = (title, url, vol_year)
        cursor.execute(sql, values)
    except Exception as e:
        print(f"Error inserting data: {e} {row['Title']}")
        break

db.commit()
cursor.close()
db.close()


In [15]:
import mysql.connector
import pandas as pd

db = mysql.connector.connect(
        host=  "127.0.0.1",
        user= "root",
        password="Nanna&143",
        database="journals",
        auth_plugin='mysql_native_password')

# Define the SQL query
query = "SELECT * FROM JAIS_DATA"

# Read data into a Pandas DataFrame
df = pd.read_sql(query, db)

# Export DataFrame to Excel
df.to_excel("JAIS_final.xlsx", index=False, engine="openpyxl")

# Close the database connection
db.close()

print("Data exported successfully to Journal_Articles.xlsx")


Data exported successfully to Journal_Articles.xlsx


/var/folders/pk/rsrc4d3x4vdgqzfqp8r7lw080000gn/T/ipykernel_1968/4183485856.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, db)
